In [2]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

In [ ]:
# bước chuẩn bị dữ liệu
%run Prepare_data.ipynb

# Các hành xây dựng mô hình Content-Based

In [4]:
def precision_at_k(recommended, actual, k=10):
    if not recommended:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    hits = len(set(recommended_k) & actual_set)
    return hits / k

def recall_at_k(recommended, actual, k=10):
    if not actual:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    hits = len(set(recommended_k) & actual_set)
    return hits / len(actual_set)

def ndcg_at_k(recommended, actual, k=10):
    if not recommended:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    dcg = 0.0
    for i, item in enumerate(recommended_k):
        if item in actual_set:
            dcg += 1.0 / np.log2(i + 2)
    ideal_hits = min(len(actual_set), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

In [5]:
def build_user_ground_truth(test_df):
    """
    Xây dựng ground truth cho bài toán user-to-item.
    Parameters:
        test_df: DataFrame chứa 'user_id', 'parent_asin', 'liked'
    Returns:
        dict: user_id -> set of parent_asin (các item liked trong test)
    """
    user_gt = test_df[test_df['liked'] == 1].groupby('user_id')['parent_asin'].apply(set).to_dict()
    return user_gt

def build_item_ground_truth(train_df, top_m=20):
    """
    Xây dựng ground truth cho bài toán item-to-item dựa trên co-occurrence.
    Parameters:
        train_df: DataFrame chứa 'user_id', 'parent_asin', 'liked' (chỉ lấy liked=1)
        top_m: số lượng item liên quan tối đa cho mỗi item (dùng để giới hạn ground truth)
    Returns:
        dict: parent_asin -> set of parent_asin (các item liên quan)
    """
    # Chỉ xét các tương tác liked=1
    liked_df = train_df[train_df['liked'] == 1]
    # Nhóm các item theo user
    user_items = liked_df.groupby('user_id')['parent_asin'].apply(set).to_dict()
    
    # Đếm số lần hai item cùng xuất hiện
    cooccur = defaultdict(Counter)
    for items in user_items.values():
        items_list = list(items)
        for i in range(len(items_list)):
            for j in range(i+1, len(items_list)):
                a, b = items_list[i], items_list[j]
                cooccur[a][b] += 1
                cooccur[b][a] += 1
    
    # Với mỗi item, lấy top_m item có số lần cùng xuất hiện cao nhất
    item_gt = {}
    for item, counter in cooccur.items():
        # Lấy top_m item (có thể ít hơn nếu không đủ)
        top_items = [it for it, _ in counter.most_common(top_m)]
        item_gt[item] = set(top_items)
    
    return item_gt

In [6]:
def evaluate_user_to_item(train_df, test_df, recommend_func, k=10):
    """
    Đánh giá mô hình user-to-item.
    """
    # Sử dụng hàm ground truth đã định nghĩa
    user_actual = build_user_ground_truth(test_df)  # thay vì viết trực tiếp
    user_interacted = train_df.groupby('user_id')['parent_asin'].apply(set).to_dict()
    
    precisions = []
    recalls = []
    ndcgs = []
    
    for user, actual in user_actual.items():
        if not actual:
            continue
        interacted = user_interacted.get(user, set())
        recommended = recommend_func(user, interacted, k)
        recommended = [item for item in recommended if item not in interacted]
        precisions.append(precision_at_k(recommended, actual, k))
        recalls.append(recall_at_k(recommended, actual, k))
        ndcgs.append(ndcg_at_k(recommended, actual, k))
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'ndcg': np.mean(ndcgs)
    }

In [7]:
def evaluate_item_to_item(train_df, item_ground_truth, recommend_func_item, k=10):
    """
    Đánh giá mô hình item-to-item.
    
    Parameters:
        train_df: không dùng trực tiếp, nhưng giữ để đồng bộ
        item_ground_truth: dict {item: set of related items}
        recommend_func_item: hàm func(item_id, k) -> list
        k: số lượng gợi ý
    """
    precisions = []
    recalls = []
    ndcgs = []
    
    for item, actual in item_ground_truth.items():
        if not actual:
            continue
        recommended = recommend_func_item(item, k)
        p = precision_at_k(recommended, actual, k)
        r = recall_at_k(recommended, actual, k)
        n = ndcg_at_k(recommended, actual, k)
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(n)
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'ndcg': np.mean(ndcgs)
    }

In [8]:
def build_item_profiles(train_df, other_features, tfidf_matrix=None, use_nlp=False):
    """
    Xây dựng item profile (vector đặc trưng cho mỗi sản phẩm).
    
    Parameters:
        train_df: DataFrame train (có cột 'parent_asin')
        other_features: list tên các cột đặc trưng định lượng/đã encode (brand_enc, store_enc, helpful_vote, average_rating, rating_number)
        tfidf_matrix: ma trận TF-IDF (sparse) của tập train, mỗi hàng tương ứng với một review.
                      Chỉ cung cấp nếu use_nlp=True.
        use_nlp: bool, nếu True thì kết hợp đặc trưng TF-IDF.
    
    Returns:
        dict: parent_asin -> vector (numpy array)
    """
    # Lấy ma trận đặc trưng cơ bản (dense)
    X_base = train_df[other_features].values  # shape (n_samples, n_base_features)
    
    # Nếu dùng NLP, ghép với ma trận TF-IDF (sparse)
    if use_nlp and tfidf_matrix is not None:
        X_combined = hstack([csr_matrix(X_base), tfidf_matrix]).tocsr()
    else:
        X_combined = X_base  # dense
    
    # Tính trung bình theo parent_asin (mỗi sản phẩm)
    item_profiles = {}
    for item, group in train_df.groupby('parent_asin'):
        indices = group.index
        if use_nlp:
            vec = X_combined[indices].mean(axis=0)
            if hasattr(vec, 'A1'):
                vec = vec.A1
        else:
            vec = X_combined[indices].mean(axis=0)
        item_profiles[item] = vec
    return item_profiles

def build_user_profiles(train_df, item_profiles):
    """
    User profile = trung bình item profile của các sản phẩm user đã thích (liked=1).
    """
    if not item_profiles:
        return {}
    dim = len(next(iter(item_profiles.values())))
    user_profiles = {}
    for user, group in train_df.groupby('user_id'):
        liked_items = group[group['liked'] == 1]['parent_asin'].tolist()
        if liked_items:
            vectors = [item_profiles[item] for item in liked_items if item in item_profiles]
            user_profiles[user] = np.mean(vectors, axis=0) if vectors else np.zeros(dim)
        else:
            user_profiles[user] = np.zeros(dim)
    return user_profiles



In [9]:
def content_recommend_user(user_id, interacted_items, user_profiles, item_profiles, k=10):
    """Gợi ý top‑k sản phẩm dựa trên cosine similarity giữa user profile và item profile."""
    user_vec = user_profiles.get(user_id)
    if user_vec is None:
        return []
    similarities = []
    for item, item_vec in item_profiles.items():
        if item in interacted_items:
            continue
        sim = cosine_similarity([user_vec], [item_vec])[0][0]
        similarities.append((item, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [item for item, _ in similarities[:k]]

def content_recommend_item(item_id, item_profiles, k=10):
    """Gợi ý top‑k sản phẩm tương tự với item_id dựa trên item profile."""
    if item_id not in item_profiles:
        return []
    target_vec = item_profiles[item_id]
    similarities = []
    for other_id, other_vec in item_profiles.items():
        if other_id == item_id:
            continue
        sim = cosine_similarity([target_vec], [other_vec])[0][0]
        similarities.append((other_id, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [item for item, _ in similarities[:k]]



# Kết quả đánh giá mô hình content-based

In [10]:


USE_NLP = True   # Lựa chọn áp dụng NLP

# --- Các cột đặc trưng cơ bản (đã được xử lý: target encoding, scaling, ...)
other_features = [
    'brand_enc', 'store_enc', 'helpful_vote', 'average_rating', 'rating_number'
]  

# --- Giả sử bạn đã có các ma trận TF-IDF từ pipeline NLP
# X_train_tfidf, X_val_tfidf, X_test_tfidf là các sparse matrix (scipy.sparse)
# Nếu không dùng NLP, bạn vẫn truyền tfidf_matrix=None và use_nlp=False

# Xây dựng item profile (dùng train)
item_profiles = build_item_profiles(
    train_df, 
    other_features, 
    tfidf_matrix=X_train if USE_NLP else None,
    use_nlp=USE_NLP
)

# Xây dựng user profile từ train
user_profiles = build_user_profiles(train_df, item_profiles)

# Hàm recommend cho user-to-item (wrapper)
def recommend_for_user(user_id, interacted, k):
    return content_recommend_user(user_id, interacted, user_profiles, item_profiles, k)

# Hàm recommend cho item-to-item
def recommend_for_item(item_id, k):
    return content_recommend_item(item_id, item_profiles, k)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, test_df, recommend_for_user, k=10)
print("=== Content-Based (User-to-item) ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_for_item, k=10)
print("\n=== Content-Based (Item-to-item) ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Content-Based (User-to-item) ===
Precision@10: 0.0008
Recall@10: 0.0078
NDCG@10: 0.0034

=== Content-Based (Item-to-item) ===
Precision@10: 0.0433
Recall@10: 0.0457
NDCG@10: 0.0634
